In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS airlinepassengers.airlinedata


In [0]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import os 
import re
connection_string= "DefaultEndpointsProtocol=https;AccountName=airlinepassenger;AccountKey=8KDl78F7zf3USzvPOVi2dMNyz3yNCKPomjyGluFbMaSqElgbeFarAec24mCdIR/uoWAP0TZfd9DJ+AStruKwcQ==;EndpointSuffix=core.windows.net"
container_name = "source"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)
for blob in container_client.list_blobs():
    print(blob.name)
for blob in container_client.list_blobs():
    table_name = (
        blob.name
        .replace("/", "_")
        .replace(" ", "_")
        .replace("-", "_")
        .replace(":", "")
        .replace("(", "")
        .replace(")", "")
        .lower()
    )
 

test.csv
train.csv


In [0]:
%sql
CREATE VOLUME airlinepassengers.airlinedata.source;

In [0]:
%sql SHOW VOLUMES IN  airlinepassengers.airlinedata

database,volume_name
airlinedata,source


In [0]:
import re

df_test = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/airlinepassengers/airlinedata/source/test.csv")

# Rename columns: replace invalid Delta characters with underscores
df_test = df_test.toDF(*[re.sub(r'[ ,;{}()\n\t=/]', '_', c) for c in df_test.columns])

df_test.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.test")
df_train=spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/airlinepassengers/airlinedata/source/train.csv")

# Rename columns: replace invalid Delta characters with underscores
df_train = df_train.toDF(*[re.sub(r'[ ,;{}()\n\t=/]', '_', c) for c in df_train.columns])

df_train.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.train")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Row
import pandas as pd
import json
import uuid
from datetime import datetime

validation_tables = [
    "airlinepassengers.airlinedata.test",
    "airlinepassengers.airlinedata.train"
]

dfs = {table.split('.')[-1]: spark.table(table) for table in validation_tables}

connection_string = "DefaultEndpointsProtocol=https;AccountName=airlinepassenger;AccountKey=8KDl78F7zf3USzvPOVi2dMNyz3yNCKPomjyGluFbMaSqElgbeFarAec24mCdIR/uoWAP0TZfd9DJ+AStruKwcQ==;EndpointSuffix=core.windows.net"
container_name = "source"

run_id = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
blob_base_path = f"validation_logs/airlinedata/run_id={run_id}"

application_keys_df = (
    dfs["test"].select("id")
    .unionByName(dfs["train"].select("id"))
    .dropDuplicates()
)

print(f"Validation run id: {run_id}")
print(f"Validation output path in blob: {blob_base_path}")
display(spark.createDataFrame([(t,) for t in validation_tables], ["table_name"]))

/root/.ipykernel/454812/command-5433020188569098-404492916:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_id = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]


Validation run id: 20260626T113730Z_b8a9a629
Validation output path in blob: validation_logs/airlinedata/run_id=20260626T113730Z_b8a9a629


table_name
airlinepassengers.airlinedata.test
airlinepassengers.airlinedata.train


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime
import json
import uuid


# Run ID
run_id = str(uuid.uuid4())


# Read tables from Unity Catalog
dfs = {
    "train": spark.table("airlinepassengers.airlinedata.train"),
    "test": spark.table("airlinepassengers.airlinedata.test")
}

validation_results = []


# Helper Function
def add_result(
    validation_id,
    validation_name,
    table_name,
    category,
    passed,
    failed_records,
    checked_records,
    details
):

    validation_results.append(
        Row(
            run_id=run_id,
            validation_id=validation_id,
            validation_name=validation_name,
            table_name=table_name,
            category=category,
            passed=bool(passed),
            failed_records=int(failed_records),
            checked_records=int(checked_records),
            failure_pct=float(
                failed_records / checked_records
                if checked_records else 0
            ),
            details=json.dumps(details),
            validated_at_utc=datetime.utcnow().isoformat()
        )
    )


# Validation Functions
def row_count_check(validation_id, table_name):

    count = dfs[table_name].count()

    add_result(
        validation_id,
        f"{table_name} Row Count > 0",
        table_name,
        "Volume",
        count > 0,
        0 if count > 0 else 1,
        max(count,1),
        {"row_count":count}
    )


def not_null_check(validation_id, table_name, column):

    df = dfs[table_name]

    checked = df.count()

    failed = df.filter(F.col(column).isNull()).count()

    add_result(
        validation_id,
        f"{column} NOT NULL",
        table_name,
        "Completeness",
        failed==0,
        failed,
        checked,
        {"column":column}
    )


def unique_check(validation_id, table_name, column):

    df = dfs[table_name]

    checked = df.count()

    distinct = df.select(column).distinct().count()

    failed = checked-distinct

    add_result(
        validation_id,
        f"{column} UNIQUE",
        table_name,
        "Uniqueness",
        failed==0,
        failed,
        checked,
        {"column":column}
    )


def allowed_values_check(validation_id, table_name, column, allowed):
    df = dfs[table_name]
    checked = df.filter(F.col(column).isNotNull()).count()
    invalid = df.filter(
        F.col(column).isNotNull() &
        (~F.col(column).isin(allowed))
    )
    failed = invalid.count()
    examples = [
        r[column]
        for r in invalid.select(column).distinct().limit(10).collect()
    ]
    add_result(
        validation_id,
        f"{column} Allowed Values",
        table_name,
        "Domain",
        failed==0,
        failed,
        max(checked,1),
        {
            "allowed_values":allowed,
            "invalid_examples":examples
        }
    )
def range_check(validation_id, table_name, column, rule, predicate):
    df = dfs[table_name]
    checked_df = df.filter(F.col(column).isNotNull())
    checked = checked_df.count()
    failed = checked_df.filter(~predicate(F.col(column))).count()
    stats = checked_df.agg(
        F.min(column).alias("min"),
        F.max(column).alias("max")
    ).collect()[0]

    add_result(
        validation_id,
        f"{column} {rule}",
        table_name,
        "Range",
        failed==0,
        failed,
        max(checked,1),
        {
            "rule":rule,
            "min":stats["min"],
            "max":stats["max"]
        }
    )


# Allowed Values
gender_values = [
    "Male",
    "Female"
]

customer_values = [
    "Loyal Customer",
    "disloyal Customer"
]

travel_values = [
    "Business travel",
    "Personal Travel"
]

class_values = [
    "Business",
    "Eco",
    "Eco Plus"
]

satisfaction_values = [
    "satisfied",
    "neutral or dissatisfied"
]

rating_columns = [
    "Inflight_wifi_service",
    "Departure_Arrival_time_convenient",
    "Ease_of_Online_booking",
    "Gate_location",
    "Food_and_drink",
    "Online_boarding",
    "Seat_comfort",
    "Inflight_entertainment",
    "On-board_service",
    "Leg_room_service",
    "Baggage_handling",
    "Checkin_service",
    "Inflight_service",
    "Cleanliness"
]

# Execute Validations
validation_id = 1

for table in ["train","test"]:

    row_count_check(validation_id,table)
    validation_id+=1

    not_null_check(validation_id,table,"id")
    validation_id+=1

    unique_check(validation_id,table,"id")
    validation_id+=1

    allowed_values_check(validation_id,table,"Gender",gender_values)
    validation_id+=1

    allowed_values_check(validation_id,table,"Customer_Type",customer_values)
    validation_id+=1

    allowed_values_check(validation_id,table,"Type_of_Travel",travel_values)
    validation_id+=1

    allowed_values_check(validation_id,table,"Class",class_values)
    validation_id+=1

    allowed_values_check(validation_id,table,"satisfaction",satisfaction_values)
    validation_id+=1

    range_check(
        validation_id,
        table,
        "Age",
        "between 0 and 120",
        lambda c:(c>=0)&(c<=120)
    )
    validation_id+=1

    range_check(
        validation_id,
        table,
        "Flight_Distance",
        ">0",
        lambda c:c>0
    )
    validation_id+=1

    range_check(
        validation_id,
        table,
        "Departure_Delay_in_Minutes",
        ">=0",
        lambda c:c>=0
    )
    validation_id+=1

    range_check(
        validation_id,
        table,
        "Arrival_Delay_in_Minutes",
        ">=0",
        lambda c:c>=0
    )
    validation_id+=1

    for column in rating_columns:

        range_check(
            validation_id,
            table,
            column,
            "between 0 and 5",
            lambda c:(c>=0)&(c<=5)
        )

        validation_id+=1

#---------------------------------------------------------
# Create Validation Results DataFrame
#---------------------------------------------------------

results_df = spark.createDataFrame(validation_results)

display(results_df)

#---------------------------------------------------------
# Save Results as Delta Table
#---------------------------------------------------------

results_df.write.mode("overwrite").saveAsTable(
    "airlinepassengers.airlinedata.validation_results"
)

print("Validation Completed Successfully.")

/root/.ipykernel/454812/command-5092096273927141-3293546928:51: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  validated_at_utc=datetime.utcnow().isoformat()


run_id,validation_id,validation_name,table_name,category,passed,failed_records,checked_records,failure_pct,details,validated_at_utc
5a1344fb-2c2f-4390-a1dd-a42528c2761b,1,train Row Count > 0,train,Volume,true,0,103904,0.0,"{""row_count"": 103904}",2026-06-26T11:50:34.062974
5a1344fb-2c2f-4390-a1dd-a42528c2761b,2,id NOT NULL,train,Completeness,true,0,103904,0.0,"{""column"": ""id""}",2026-06-26T11:50:34.645042
5a1344fb-2c2f-4390-a1dd-a42528c2761b,3,id UNIQUE,train,Uniqueness,true,0,103904,0.0,"{""column"": ""id""}",2026-06-26T11:50:35.831677
5a1344fb-2c2f-4390-a1dd-a42528c2761b,4,Gender Allowed Values,train,Domain,true,0,103904,0.0,"{""allowed_values"": [""Male"", ""Female""], ""invalid_examples"": []}",2026-06-26T11:50:36.907066
5a1344fb-2c2f-4390-a1dd-a42528c2761b,5,Customer_Type Allowed Values,train,Domain,true,0,103904,0.0,"{""allowed_values"": [""Loyal Customer"", ""disloyal Customer""], ""invalid_examples"": []}",2026-06-26T11:50:37.906519
5a1344fb-2c2f-4390-a1dd-a42528c2761b,6,Type_of_Travel Allowed Values,train,Domain,true,0,103904,0.0,"{""allowed_values"": [""Business travel"", ""Personal Travel""], ""invalid_examples"": []}",2026-06-26T11:50:39.761310
5a1344fb-2c2f-4390-a1dd-a42528c2761b,7,Class Allowed Values,train,Domain,true,0,103904,0.0,"{""allowed_values"": [""Business"", ""Eco"", ""Eco Plus""], ""invalid_examples"": []}",2026-06-26T11:50:40.965226
5a1344fb-2c2f-4390-a1dd-a42528c2761b,8,satisfaction Allowed Values,train,Domain,true,0,103904,0.0,"{""allowed_values"": [""satisfied"", ""neutral or dissatisfied""], ""invalid_examples"": []}",2026-06-26T11:50:43.408266
5a1344fb-2c2f-4390-a1dd-a42528c2761b,9,Age between 0 and 120,train,Range,true,0,103904,0.0,"{""rule"": ""between 0 and 120"", ""min"": 7, ""max"": 85}",2026-06-26T11:50:44.802361
5a1344fb-2c2f-4390-a1dd-a42528c2761b,10,Flight_Distance >0,train,Range,true,0,103904,0.0,"{""rule"": "">0"", ""min"": 31, ""max"": 4983}",2026-06-26T11:50:45.857698


Validation Completed Successfully.


In [0]:
from typing import Callable, Dict, List

allowed_bureau_balance_status = ['0', '1', '2', '3', '4', '5', 'C', 'X']

validation_results = []


def add_result(validation_id, validation_name, table_name, category, passed, failed_records, checked_records, details):
    validation_results.append(
        Row(
            run_id=run_id,
            validation_id=validation_id,
            validation_name=validation_name,
            table_name=table_name,
            category=category,
            passed=bool(passed),
            failed_records=int(failed_records),
            checked_records=int(checked_records),
            failure_pct=float((failed_records / checked_records) if checked_records else 0.0),
            details=json.dumps(details, default=str),
            validated_at_utc=datetime.utcnow().isoformat()
        )
    )


def row_count_check(validation_id, table_name):
    checked_records = dfs[table_name].count()
    add_result(
        validation_id,
        f"{table_name} row count > 0",
        table_name,
        "volume",
        checked_records > 0,
        0 if checked_records > 0 else 1,
        max(checked_records, 1),
        {"row_count": checked_records}
    )


def not_null_check(validation_id, table_name, column_name):
    df = dfs[table_name]
    checked_records = df.count()
    failed_records = df.filter(F.col(column_name).isNull()).count()
    add_result(
        validation_id,
        f"{table_name}.{column_name} has no nulls",
        table_name,
        "completeness",
        failed_records == 0,
        failed_records,
        checked_records,
        {"column_name": column_name}
    )


def unique_check(validation_id, table_name, column_name):
    df = dfs[table_name]
    checked_records = df.count()
    distinct_records = df.select(column_name).distinct().count()
    failed_records = checked_records - distinct_records
    add_result(
        validation_id,
        f"{table_name}.{column_name} is unique",
        table_name,
        "uniqueness",
        failed_records == 0,
        failed_records,
        checked_records,
        {"column_name": column_name, "distinct_records": distinct_records}
    )


def allowed_values_check(validation_id, table_name, column_name, allowed_values):
    df = dfs[table_name]
    checked_records = df.filter(F.col(column_name).isNotNull()).count()
    failed_records = df.filter(F.col(column_name).isNotNull() & (~F.col(column_name).isin(allowed_values))).count()
    invalid_examples = [row[column_name] for row in df.filter(F.col(column_name).isNotNull() & (~F.col(column_name).isin(allowed_values))).select(column_name).distinct().limit(10).collect()]
    add_result(
        validation_id,
        f"{table_name}.{column_name} values are allowed",
        table_name,
        "domain",
        failed_records == 0,
        failed_records,
        max(checked_records, 1),
        {"column_name": column_name, "allowed_values": allowed_values, "invalid_examples": invalid_examples}
    )


def range_check(validation_id, table_name, column_name, predicate_description, predicate):
    df = dfs[table_name]
    checked_df = df.filter(F.col(column_name).isNotNull())
    checked_records = checked_df.count()
    failed_records = checked_df.filter(~predicate(F.col(column_name))).count()
    stats = checked_df.agg(
        F.min(F.col(column_name)).alias('min_value'),
        F.max(F.col(column_name)).alias('max_value')
    ).collect()[0]
    add_result(
        validation_id,
        f"{table_name}.{column_name} {predicate_description}",
        table_name,
        "range",
        failed_records == 0,
        failed_records,
        max(checked_records, 1),
        {"column_name": column_name, "rule": predicate_description, "min_value": stats['min_value'], "max_value": stats['max_value']}
    )


def date_parse_check(validation_id, table_name, column_name):
    df = dfs[table_name]
    checked_df = df.filter(F.col(column_name).isNotNull())
    checked_records = checked_df.count()
    parsed_col = F.coalesce(
        F.to_date(F.col(column_name), 'yyyy-MM-dd'),
        F.to_date(F.col(column_name), 'MM/dd/yyyy'),
        F.to_date(F.col(column_name), 'dd-MM-yyyy')
    )
    failed_records = checked_df.filter(parsed_col.isNull()).count()
    sample_bad_values = [row[column_name] for row in checked_df.filter(parsed_col.isNull()).select(column_name).distinct().limit(10).collect()]
    add_result(
        validation_id,
        f"{table_name}.{column_name} is parseable as a date",
        table_name,
        "format",
        failed_records == 0,
        failed_records,
        max(checked_records, 1),
        {"column_name": column_name, "sample_bad_values": sample_bad_values}
    )


def foreign_key_check(validation_id, child_table_name, child_column, parent_df, parent_column, check_name):
    child_df = dfs[child_table_name].select(child_column).filter(F.col(child_column).isNotNull())
    checked_records = child_df.count()
    failed_records = (
        child_df.alias('child')
        .join(parent_df.select(parent_column).dropDuplicates().alias('parent'), F.col(f'child.{child_column}') == F.col(f'parent.{parent_column}'), 'left_anti')
        .count()
    )
    add_result(
        validation_id,
        check_name,
        child_table_name,
        "referential_integrity",
        failed_records == 0,
        failed_records,
        max(checked_records, 1),
        {"child_column": child_column, "parent_column": parent_column}
    )

In [0]:

# Row Count Checks
row_count_check(1, "train")
row_count_check(2, "test")

# Not Null Checks
not_null_check(3, "train", "id")
not_null_check(4, "test", "id")

# Unique Checks
unique_check(5, "train", "id")
unique_check(6, "test", "id")

# Allowed Values Checks
allowed_values_check(
    7,
    "train",
    "Gender",
    ["Male", "Female"]
)

allowed_values_check(
    8,
    "test",
    "Gender",
    ["Male", "Female"]
)

allowed_values_check(
    9,
    "train",
    "Customer_Type",
    ["Loyal Customer", "disloyal Customer"]
)

allowed_values_check(
    10,
    "test",
    "Customer_Type",
    ["Loyal Customer", "disloyal Customer"]
)

allowed_values_check(
    11,
    "train",
    "Type_of_Travel",
    ["Business travel", "Personal Travel"]
)

allowed_values_check(
    12,
    "test",
    "Type_of_Travel",
    ["Business travel", "Personal Travel"]
)

allowed_values_check(
    13,
    "train",
    "Class",
    ["Business", "Eco", "Eco Plus"]
)

allowed_values_check(
    14,
    "test",
    "Class",
    ["Business", "Eco", "Eco Plus"]
)

allowed_values_check(
    15,
    "train",
    "satisfaction",
    ["satisfied", "neutral or dissatisfied"]
)

allowed_values_check(
    16,
    "test",
    "satisfaction",
    ["satisfied", "neutral or dissatisfied"]
)

# Range Checks
range_check(
    17,
    "train",
    "Age",
    "between 0 and 120",
    lambda c: (c >= F.lit(0)) & (c <= F.lit(120))
)

range_check(
    18,
    "test",
    "Age",
    "between 0 and 120",
    lambda c: (c >= F.lit(0)) & (c <= F.lit(120))
)

range_check(
    19,
    "train",
    "Flight_Distance",
    "greater than 0",
    lambda c: c > F.lit(0)
)

range_check(
    20,
    "test",
    "Flight_Distance",
    "greater than 0",
    lambda c: c > F.lit(0)
)

range_check(
    21,
    "train",
    "Departure_Delay_in_Minutes",
    "greater than or equal to 0",
    lambda c: c >= F.lit(0)
)

range_check(
    22,
    "test",
    "Departure_Delay_in_Minutes",
    "greater than or equal to 0",
    lambda c: c >= F.lit(0)
)

range_check(
    23,
    "train",
    "Arrival_Delay_in_Minutes",
    "greater than or equal to 0",
    lambda c: c >= F.lit(0)
)

range_check(
    24,
    "test",
    "Arrival_Delay_in_Minutes",
    "greater than or equal to 0",
    lambda c: c >= F.lit(0)
)

# Rating Columns (0-5)
rating_columns = [
    "Inflight_wifi_service",
    "Departure_Arrival_time_convenient",
    "Ease_of_Online_booking",
    "Gate_location",
    "Food_and_drink",
    "Online_boarding",
    "Seat_comfort",
    "Inflight_entertainment",
    "On-board_service",
    "Leg_room_service",
    "Baggage_handling",
    "Checkin_service",
    "Inflight_service",
    "Cleanliness"
]

validation_id = 25

for table in ["train", "test"]:

    for column in rating_columns:

        range_check(
            validation_id,
            table,
            column,
            "between 0 and 5",
            lambda c: (c >= F.lit(0)) & (c <= F.lit(5))
        )

        validation_id += 1

# Create Validation DataFrames
validation_results_df = spark.createDataFrame(validation_results).orderBy("validation_id")

validation_summary_df = (
    validation_results_df
    .groupBy("run_id")
    .agg(
        F.count("*").alias("total_validations"),
        F.sum(F.when(F.col("passed"), 1).otherwise(0)).alias("passed_validations"),
        F.sum(F.when(~F.col("passed"), 1).otherwise(0)).alias("failed_validations")
    )
)

failed_validations_df = (
    validation_results_df
    .filter(~F.col("passed"))
    .orderBy("validation_id")
)

print("Detailed Validation Results")
display(validation_results_df)

print("Validation Summary")
display(validation_summary_df)

print("Failed Validations")
display(failed_validations_df)

/root/.ipykernel/454812/command-5092096273927141-3293546928:51: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  validated_at_utc=datetime.utcnow().isoformat()


Detailed Validation Results


run_id,validation_id,validation_name,table_name,category,passed,failed_records,checked_records,failure_pct,details,validated_at_utc
5a1344fb-2c2f-4390-a1dd-a42528c2761b,1,train Row Count > 0,train,Volume,true,0,103904,0.0,"{""row_count"": 103904}",2026-06-26T12:55:26.437305
5a1344fb-2c2f-4390-a1dd-a42528c2761b,1,train Row Count > 0,train,Volume,true,0,103904,0.0,"{""row_count"": 103904}",2026-06-26T11:50:34.062974
5a1344fb-2c2f-4390-a1dd-a42528c2761b,1,train Row Count > 0,train,Volume,true,0,103904,0.0,"{""row_count"": 103904}",2026-06-26T12:53:40.175970
5a1344fb-2c2f-4390-a1dd-a42528c2761b,2,id NOT NULL,train,Completeness,true,0,103904,0.0,"{""column"": ""id""}",2026-06-26T11:50:34.645042
5a1344fb-2c2f-4390-a1dd-a42528c2761b,2,test Row Count > 0,test,Volume,true,0,25976,0.0,"{""row_count"": 25976}",2026-06-26T12:55:26.834983
5a1344fb-2c2f-4390-a1dd-a42528c2761b,2,test Row Count > 0,test,Volume,true,0,25976,0.0,"{""row_count"": 25976}",2026-06-26T12:53:41.163419
5a1344fb-2c2f-4390-a1dd-a42528c2761b,3,id UNIQUE,train,Uniqueness,true,0,103904,0.0,"{""column"": ""id""}",2026-06-26T11:50:35.831677
5a1344fb-2c2f-4390-a1dd-a42528c2761b,3,id NOT NULL,train,Completeness,true,0,103904,0.0,"{""column"": ""id""}",2026-06-26T12:55:27.253182
5a1344fb-2c2f-4390-a1dd-a42528c2761b,3,id NOT NULL,train,Completeness,true,0,103904,0.0,"{""column"": ""id""}",2026-06-26T12:53:41.675343
5a1344fb-2c2f-4390-a1dd-a42528c2761b,4,Gender Allowed Values,train,Domain,true,0,103904,0.0,"{""allowed_values"": [""Male"", ""Female""], ""invalid_examples"": []}",2026-06-26T11:50:36.907066


Validation Summary


run_id,total_validations,passed_validations,failed_validations
5a1344fb-2c2f-4390-a1dd-a42528c2761b,112,112,0


Failed Validations


run_id,validation_id,validation_name,table_name,category,passed,failed_records,checked_records,failure_pct,details,validated_at_utc


In [0]:
validation_results_df = spark.createDataFrame(validation_results).orderBy("validation_id")

validation_summary_df = (
    validation_results_df.groupBy("run_id")
    .agg(
        F.count("*").alias("total_validations"),
        F.sum(F.when(F.col("passed"), 1).otherwise(0)).alias("passed_validations"),
        F.sum(F.when(~F.col("passed"), 1).otherwise(0)).alias("failed_validations")
    )
)

failed_validations_df = validation_results_df.filter(~F.col("passed"))

In [0]:
validation_results_table = "airlinepassengers.airlinedata.validation_results"
validation_summary_table = "airlinepassengers.airlinedata.validation_summary"
validation_failed_table = "airlinepassengers.airlinedata.validation_failed_results"

validation_results_df.write.mode("append").saveAsTable(validation_results_table)
validation_summary_df.write.mode("append").saveAsTable(validation_summary_table)
failed_validations_df.write.mode("append").saveAsTable(validation_failed_table)

In [0]:
artifact_rows = [
    ("run_id", run_id),
    ("blob_base_path", blob_base_path),
    ("results_table", validation_results_table),
    ("summary_table", validation_summary_table),
    ("failed_table", validation_failed_table)
]

artifacts_df = spark.createDataFrame(
    artifact_rows,
    ["artifact_type", "artifact_location"]
)

display(artifacts_df)

artifact_type,artifact_location
run_id,5a1344fb-2c2f-4390-a1dd-a42528c2761b
blob_base_path,validation_logs/airlinedata/run_id=20260626T113730Z_b8a9a629
results_table,airlinepassengers.airlinedata.validation_results
summary_table,airlinepassengers.airlinedata.validation_summary
failed_table,airlinepassengers.airlinedata.validation_failed_results


In [0]:
#pipeline logging
#pipeline_start_time
from datetime import datetime
import uuid
run_id = str(uuid.uuid4())

pipeline_start_time = datetime.utcnow()
pipeline_end_time = None

processing_status = "STARTED"
error_message = None

/root/.ipykernel/454812/command-8659297391342505-1445000655:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  pipeline_start_time = datetime.utcnow()


In [0]:
try:

    # Read train.csv
    train_df = spark.read.option("header", "true").csv(
        "/Volumes/airlinepassengers/airlinedata/source/train.csv"
    )

    # Read test.csv
    test_df = spark.read.option("header", "true").csv(
        "/Volumes/airlinepassengers/airlinedata/source/test.csv"
    )

    dfs = {
        "train": train_df,
        "test": test_df
    }

    processing_status = "SUCCESS"
except Exception as e:

    processing_status = "FAILED"
    error_message = str(e)

finally:

    pipeline_end_time = datetime.utcnow()

/root/.ipykernel/454812/command-6037312457376199-3251046700:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  pipeline_end_time = datetime.utcnow()


In [0]:
from pyspark.sql import Row

execution_log = [
    Row(
        run_id=run_id,
        pipeline_start_time=pipeline_start_time,
        pipeline_end_time=pipeline_end_time,
        processing_status=processing_status,
        error_message=error_message
    )
]

from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType
schema = StructType([
    StructField('run_id', StringType(), True),
    StructField('pipeline_start_time', TimestampType(), True),
    StructField('pipeline_end_time', TimestampType(), True),
    StructField('processing_status', StringType(), True),
    StructField('error_message', StringType(), True)
])

execution_log_df = spark.createDataFrame(execution_log, schema=schema)


display(execution_log_df)

run_id,pipeline_start_time,pipeline_end_time,processing_status,error_message
93d7f8b5-dfb3-4190-a4bb-efec04a1a029,2026-06-26T14:48:20.639651Z,2026-06-26T14:48:21.647929Z,SUCCESS,null


In [0]:
execution_log_table = "airlinepassengers.airlinedata.execution_log"

execution_log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(execution_log_table)

In [0]:
from pyspark.sql import Row
from datetime import datetime
import uuid

# Batch ID
batch_id = str(uuid.uuid4())

# Processing Date
processing_date = datetime.now().date()

# Ingestion Timestamp
ingestion_timestamp = datetime.utcnow()

audit_rows = [

    Row(
        run_id=run_id,
        batch_id=batch_id,
        source_file_name="train.csv",
        ingestion_timestamp=ingestion_timestamp,
        processing_date=processing_date,
        record_count=train_df.count()
    ),

    Row(
        run_id=run_id,
        batch_id=batch_id,
        source_file_name="test.csv",
        ingestion_timestamp=ingestion_timestamp,
        processing_date=processing_date,
        record_count=test_df.count()
    )

]

audit_df = spark.createDataFrame(audit_rows)

display(audit_df)

/root/.ipykernel/454812/command-6037312457376205-1680533462:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ingestion_timestamp = datetime.utcnow()


run_id,batch_id,source_file_name,ingestion_timestamp,processing_date,record_count
93d7f8b5-dfb3-4190-a4bb-efec04a1a029,278b5530-4aa5-465d-ac93-09b3bce2d9e2,train.csv,2026-06-26T14:33:36.938931Z,2026-06-26,103904
93d7f8b5-dfb3-4190-a4bb-efec04a1a029,278b5530-4aa5-465d-ac93-09b3bce2d9e2,test.csv,2026-06-26T14:33:36.938931Z,2026-06-26,25976


In [0]:
audit_table = "airlinepassengers.airlinedata.audit_log"

audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(audit_table)

In [0]:
display(
    spark.table("airlinepassengers.airlinedata.audit_log")
)

run_id,batch_id,source_file_name,ingestion_timestamp,processing_date,record_count
93d7f8b5-dfb3-4190-a4bb-efec04a1a029,278b5530-4aa5-465d-ac93-09b3bce2d9e2,train.csv,2026-06-26T14:33:36.938931Z,2026-06-26,103904
93d7f8b5-dfb3-4190-a4bb-efec04a1a029,278b5530-4aa5-465d-ac93-09b3bce2d9e2,test.csv,2026-06-26T14:33:36.938931Z,2026-06-26,25976


In [0]:
#1. Validate File Name
import os

expected_files = ["train.csv", "test.csv"]
volume_path = "/Volumes/airlinepassengers/airlinedata/source/"
actual_files = [f.name for f in dbutils.fs.ls(volume_path)]

for file in expected_files:
    if file in actual_files:
        print(f"PASS : {file} exists.")
    else:
        raise Exception(f"FAIL : {file} is missing.")

PASS : train.csv exists.
PASS : test.csv exists.


In [0]:
#2. Check File Size > 0
files = dbutils.fs.ls("/Volumes/airlinepassengers/airlinedata/source/")
for file in files:
    if file.size > 0:
        print(f"PASS : {file.name} Size = {file.size} bytes")
    else:
        raise Exception(f"FAIL : {file.name} is empty.")

PASS : test.csv Size = 3037688 bytes
PASS : train.csv Size = 12193089 bytes


In [0]:
#Duplicate File check
source_file = "train.csv"
duplicate_count = spark.sql(f"""
SELECT COUNT(*)
FROM airlinepassengers.airlinedata.audit_log
WHERE source_file_name = '{source_file}'
AND run_id != '{run_id}'
""").collect()[0][0]
if duplicate_count > 0:
    raise Exception(f"{source_file} has already been ingested.")
print("No duplicate file found.")

No duplicate file found.


In [0]:
files = ["train.csv", "test.csv"]

for file in files:

    duplicate = spark.sql(f"""
        SELECT COUNT(*)
        FROM airlinepassengers.airlinedata.audit_log
        WHERE source_file_name = '{file}'
    """).collect()[0][0]

    if duplicate > 0:
        print(f"Duplicate Found : {file}")

    else:
        print(f"No Duplicate : {file}")

Duplicate Found : train.csv
Duplicate Found : test.csv


In [0]:
from datetime import datetime, UTC
pipeline_start_time =datetime.now(UTC)
processing_status = "STARTED"
error_message = None

try:
    train_df = spark.read.option("header","true").csv(
        "/Volumes/airlinepassengers/airlinedata/source/train.csv"
    )
    test_df = spark.read.option("header","true").csv(
        "/Volumes/airlinepassengers/airlinedata/source/test.csv"
    )
    processing_status = "SUCCESS"
except Exception as e:
    processing_status = "FAILED"
    error_message = str(e)
finally:
    pipeline_end_time = datetime.now(UTC)
print("Pipeline Status :", processing_status)
print("Error :", error_message)

Pipeline Status : SUCCESS
Error : None
